# 🧪 Module 1 Lab: Hugging Face Pipelines (Exercises)

Welcome to your first hands-on Lab!

---
> 🎓 **INSTRUCTOR NOTES & LAB FACILITATION GUIDE**
>
> **Target Duration:** ~30-45 minutes
>
> **Lab Objectives:**
> - Ensure students can construct `pipeline()` calls independently.
> - Verify students understand filtering results based on probability thresholds.
> - Practice custom label engineering in Zero-Shot pipelines.


In [1]:
# Import required modules
import torch
from transformers import pipeline

device_id = 0 if torch.cuda.is_available() else -1
print(f"Lab Environment Ready! Execution Device: {'GPU' if device_id==0 else 'CPU'}")



Lab Environment Ready! Execution Device: GPU


### 🏋️ Challenge 1: High-Confidence Sentiment Review Classifier
**Goal:** Create a sentiment analysis pipeline and evaluate a list of 4 user reviews.
Print only reviews that have a `POSITIVE` label AND a confidence score higher than `0.90`.


In [25]:
reviews = [
    "The battery life on this laptop is incredible, lasts all day!",
    "Extremely disappointed. Screen cracked within two days of normal use.",
    "Decent performance for the price, though the keyboard feels cheap.",
    "Unbelievable customer service and ultra fast delivery. 10/10!"
]
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", device=device_id)
for review in reviews:
    sentiment = sentiment_pipeline(review)[0]
    if sentiment['label'] == 'POSITIVE' and sentiment['score'] > 0.90:
        print(review)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

The battery life on this laptop is incredible, lasts all day!
Decent performance for the price, though the keyboard feels cheap.


### 🏋️ Challenge 2: Zero-Shot Support Ticket Router
**Goal:** Categorize customer tickets into 4 business categories: `"Refund & Returns"`, `"Bug Report"`, `"Feature Request"`, `"Account Security"`.


In [23]:
tickets = [
    "I saw a suspicious login attempt from another country on my account.",
    "The app crashes every time I click on the export CSV button.",
    "Can you please add a Dark Mode option to the desktop application?",
    "I bought the wrong size shirt and want to send it back for a full refund."
]

departments = ["Refund & Returns", "Bug Report", "Feature Request", "Account Security"]

zero_shot_caregorize = pipeline("zero-shot-classification",model="cross-encoder/nli-deberta-v3-base", device=device_id)

results = zero_shot_caregorize(
    tickets,
    candidate_labels=departments,
    hypothesis_template="This ticket is about {}."
)

for ticket, result in zip(tickets, results):
    top_label = result['labels'][0]
    confidence_score = result['scores'][0]

    print(f"Ticket: \"{ticket}\"")
    print(f"Categorized as: [{top_label}] (Confidence: {confidence_score:.2%})\n")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Ticket: "I saw a suspicious login attempt from another country on my account."
Categorized as: [Account Security] (Confidence: 75.48%)

Ticket: "The app crashes every time I click on the export CSV button."
Categorized as: [Bug Report] (Confidence: 86.89%)

Ticket: "Can you please add a Dark Mode option to the desktop application?"
Categorized as: [Feature Request] (Confidence: 99.14%)

Ticket: "I bought the wrong size shirt and want to send it back for a full refund."
Categorized as: [Refund & Returns] (Confidence: 97.25%)



### 🏋️ Challenge 3: Entity Extractor for News Paragraphs
**Goal:** Extract entities from the text and count how many **Locations (LOC)** are mentioned.


In [24]:
news_text = """
Google announced new AI research centers in London and Tokyo, while Microsoft expanded its partnership with OpenAI in San Francisco. CEO Sundar Pichai stated this initiative will boost global research.
"""

entity_extractor = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english", aggregation_strategy="simple", device=device_id)

entities = entity_extractor(news_text)

locations = [entity['word'] for entity in entities if entity['entity_group'] == 'LOC']
location_count = len(locations)
print(f"Number of Locations (LOC) mentioned: {location_count}")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of Locations (LOC) mentioned: 3
